Assume the latent factor model 
$$
\mathbf{Y}_t = \boldsymbol{\Theta}_t\mathbf{Z} + \boldsymbol{\Lambda}_t \mathbf{M},
$$ 
where $t\in 1,\ldots, T$ and the matrices $\boldsymbol{\Theta}_t$ and $\boldsymbol{\Lambda}_t$ are:
$$
\begin{pmatrix}
\theta_{t_{11}}&\dots &\theta_{t_{1p}} \\
\vdots & & \vdots \\
\theta_{t_{m1}} &\dots&\theta_{t_{mp}}
\end{pmatrix} \quad\text{and}\quad
\begin{pmatrix}
\lambda_{t_{11}}&\dots &\lambda_{t_{1f}} \\
\vdots & & \vdots \\
\lambda_{t_{m1}} &\dots&\lambda_{t_{mf}}
\end{pmatrix}
$$ 
and the matrices $\mathbf{Z}$ and $\mathbf{M}$ are
$$
\begin{pmatrix}
Z_{11}&\dots & Z_{N-1,1} \\
\vdots & & \vdots \\
Z_{1p} &\dots& Z_{N-1, p}
\end{pmatrix} \quad\text{and}\quad
\begin{pmatrix}
\mu_{11}&\dots & \mu_{N-1,1} \\
\vdots & & \vdots \\
\mu_{1p} &\dots& \mu_{N-1, p}
\end{pmatrix}
$$ 

In [56]:
a = [1, 2, 3]
a[len(a)-1]
a[0:len(a)-1]

[1, 2]

In [113]:
import numpy as np
import torch
import learnQ as lq

torch.manual_seed(42)
np.random.seed(42)

T = 2
N = 3
m = 3
p = 4
f = 5

Outcomes = []
Thetas = []
Lambdas = []

# Fixed base loadings
Theta_base = torch.randn(m, p, dtype=torch.float64)
Theta_step = torch.randn(m, p, dtype=torch.float64) * 1
Lambda_base = torch.randn(m, f, dtype=torch.float64)
Lambda_step = torch.randn(m, f, dtype=torch.float64) * 1

for t in range(T):
    Theta_t = Theta_base + Theta_step * torch.randn(m, p, dtype=torch.float64)
    Lambda_t = Lambda_base + Lambda_step * torch.randn(m, f, dtype=torch.float64)
    Thetas.append(Theta_t)
    Lambdas.append(Lambda_t)

Z_donors = torch.randn(p, N, dtype=torch.float64)   # (p x N)
mu_donors = torch.randn(f, N, dtype=torch.float64)  # (f x N)

for t in range(T):
    Y_t = Thetas[t] @ Z_donors + Lambdas[t] @ mu_donors
    Y_t += 0 * torch.randn_like(Y_t)
    Outcomes.append(Y_t)

# for t in range(T):
#     print(f"Time {t}:\n", Outcomes[t].round(decimals=2), "\n")  # Should print (m, N) for each time period

targets = []
covariates = []
for outcome in Outcomes:
    # take the first column
    targets.append(outcome[:, 0])
    # all but the first column
    covariates.append(outcome[:, 1:])

Q_final, w_final = lq.learnQ(targets, covariates, embedding_dim=5, n_iterations=2000, reg_Q=0, reg_w=1, verbose=True)

# print("Synthetic Covariates: \n", covariates[len(covariates)-1] @ Q_final)
# print("True Covariates: \n", covariates[len(covariates)-1])
# print("Synthetic prediction:\n", covariates[len(covariates)-1] @ Q_final @ w_final)
# print("True target:\n", targets[len(targets)-1])




Step    0 | Loss: 22.59879536
Step    0 | Loss: 22.59879536 | w: [0.368 0.198 0.088 0.187 0.159]
Grad norm: 0.05336855
Step  200 | Loss: 22.62519684
Step  200 | Loss: 22.62519684 | w: [0.    0.218 0.161 0.346 0.275]
Grad norm: 0.00000112
Step  400 | Loss: 22.62520218
Step  400 | Loss: 22.62520218 | w: [0.    0.218 0.161 0.346 0.275]
Grad norm: 0.00000001
Step  600 | Loss: 22.62520722
Step  600 | Loss: 22.62520722 | w: [0.    0.218 0.161 0.346 0.275]
Grad norm: 0.00000001
Step  800 | Loss: 22.62521151
Step  800 | Loss: 22.62521151 | w: [0.    0.218 0.161 0.346 0.275]
Grad norm: 0.00000001
Step 1000 | Loss: 22.62521665
Step 1000 | Loss: 22.62521665 | w: [0.    0.218 0.161 0.346 0.275]
Grad norm: 0.00000001
Step 1200 | Loss: 22.62521966
Step 1200 | Loss: 22.62521966 | w: [0.    0.218 0.161 0.346 0.275]
Grad norm: 0.00000001
Step 1400 | Loss: 22.62522314
Step 1400 | Loss: 22.62522314 | w: [0.    0.218 0.161 0.346 0.275]
Grad norm: 0.00000001
Step 1600 | Loss: 22.62522608
Step 1600 | Loss: 

In [114]:
# Compute time-averaged Gram matrix
gram_sum = sum(A.numpy().T @ A.numpy() for A in covariates[0:len(covariates)-1]) / (T-1)
# (n x n) = (9 x 9)

D = Q_final.shape[1]

eigenvalues, eigenvectors = np.linalg.eigh(gram_sum)
eigenvectors = eigenvectors[:, ::-1]   # flip to descending order
V_gram = eigenvectors[:, :D]           # leading D eigenvectors, (n x D)

Q_orth, _ = np.linalg.qr(Q_final)     # (n x D)
print("Q_orth is square:", Q_orth.shape[0] == Q_orth.shape[1])

cos_gram = np.linalg.svd(Q_orth.T @ V_gram, compute_uv=False)  # (D x D) -> D singular values
print("Alignment with time-averaged Gram:", cos_gram.round(3))


diff = [((A.numpy().T @ A.numpy() @ (Q_final @ w_final)) - (A.numpy().T @ d.numpy())) for A, d in zip(covariates, targets)]
print(diff)
print(np.mean(diff))

Q_orth is square: True
Alignment with time-averaged Gram: [1. 1.]
[array([-17.60359738,  -3.54640139]), array([17.604097  ,  3.54647264])]
0.00014271609008686958
